# Compare players at the same number of games

**Three fresh alternatives:** MLP, GRU and larger Transformer. Compare them with your **preserved 19,931-game small Transformer**, both with and without bounded combat look-ahead. We do not retrain or replace your existing player.

- **Target: 19,931 completed training games per alternative.** Graphs update at **1,000, 2,000, … 19,000**, and the final 19,931. Each shared checkpoint uses exactly the same completed-game count; time is measured, not matched.
- **Graphs:** win rate against common opponents, tactical-question accuracy and training hours versus games. Reference lines are the already-trained saved player, not its early learning curve. PNG and SVG copies are saved for sharing.
- **One launch: two-hour soft session limit.** Active work/checkpoint saving may finish afterward. Run All Cells again later to resume all saved progress; there is no automatic multi-day run.
- **Same alternative training setup:** all 176 decks/eleven classes, six learning threads, four collectors, 60/30/10 current self-play/older-model/heuristic mix, same PPO and tactical teaching.
- GRU memory spans sixteen own decisions and resets per game. The fresh-study league retains eight checkpoints; common PPO minibatches are sixteen for memory safety. These differ from parts of the saved player's training history, which you chose to accept.
- **Search:** the preserved small Transformer plus two-ply plain-minion combat planning. It never reads private hands or deck order. Unsupported positions use the base policy; coverage/extra time are reported. It is not a full-game planner and does not model unknown secret activation.
- Errors/caps are reported, not counted as losses or used for learning. Complete, valid shared checkpoints are required before ranking alternatives. This first study uses one seed; it is a pilot, not a publication-grade architecture conclusion.

**Run All Cells when ready.** The first code cell only checks the setup. The second starts the owner-controlled session. **Nothing has been started during preparation.** Notebook 21 and your saved model are unchanged; the shared lock prevents simultaneous training.

More strength per game suggests better sample efficiency. More seconds per game shows its compute cost. This study alone cannot establish how much faster a different computer would be.


In [ ]:
from pathlib import Path
import json, subprocess
from IPython.display import HTML, display
ROOT=next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
PYTHON=ROOT/'.venv/bin/python'
TOOL=ROOT/'tools/run_matched_games_comparison.py'
PREFLIGHT_OK=False
check=subprocess.run([str(PYTHON),str(TOOL)],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(check.stdout)
if check.returncode: raise RuntimeError('Research checks failed; nothing was started.')
PREFLIGHT_OK=True
print('Ready. No training started. The next cell starts the two-hour research session.')


In [ ]:
# OWNER START: run this cell only when you want the research session to begin.
if not globals().get('PREFLIGHT_OK',False): raise RuntimeError('Run the checks first.')
import time, os, signal
last_redraw=0.0
screen=display(HTML('<h2>Preparing the research session…</h2>'),display_id=True)
with subprocess.Popen([str(PYTHON),str(TOOL),'--train'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=True) as process:
    try:
        for line in process.stdout:
            try: event=json.loads(line)
            except ValueError: print(line.rstrip());continue
            if event.get('event') in ('study_started','checkpoint','evaluation_complete','session_complete'):
                print(json.dumps(event))
            now=time.monotonic()
            if now-last_redraw>=15 or event.get('event') in ('checkpoint','evaluation_complete','session_complete'):
                progress=event.get('arm','Preparing')
                status=event.get('event','').replace('_',' ')
                done=event.get('games')
                note=f' · {done:,} games' if done is not None else ''
                report=ROOT/'runs/research_matched_games/dashboard.html'
                content=report.read_text() if report.exists() else '<p>No completed evaluations yet. Learners are starting fresh.</p>'
                screen.update(HTML(f'<h2>{progress}: {status}{note}</h2>'+content));last_redraw=now
        code=process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGTERM)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: os.killpg(process.pid,signal.SIGKILL);process.wait()
        raise
if code: raise RuntimeError('Research stopped. Its last durable checkpoints and error receipts remain saved; inspect the message above.')
print('Session finished. Run All Cells again later to resume; original notebook 21/model are unchanged.')
